In [26]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
import string
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense,GRU,SimpleRNN,Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report,confusion_matrix
from sklearn.preprocessing import LabelEncoder


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [27]:
df=pd.read_csv('IMDB Dataset.csv')

In [28]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [29]:
df.isnull().sum()

,0
review,0
sentiment,0


In [30]:
df.duplicated().sum()

np.int64(418)

In [31]:
df=df.drop_duplicates()

In [32]:
df['sentiment'].value_counts(normalize=True)

,proportion
sentiment,
positive,0.501876
negative,0.498124


In [33]:
def nlp_preprocess(text):
    # Convert to lowercase
    text = text.lower()

    # Remove hyperlinks (URLs)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Remove HTML tags (common in IMDB dataset)
    text = re.sub(r'<.*?>', '', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Tokenization
    tokens = word_tokenize(text)

    # Remove stopwords and apply Lemmatization
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()

    cleaned_text = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]

    return " ".join(cleaned_text)

# Apply the function to the dataset
df['cleaned_review'] = df['review'].apply(nlp_preprocess)
display(df.head())

,review,sentiment,cleaned_review
0,One of the other reviewers has mentioned that ...,positive,one reviewer mentioned watching 1 oz episode y...
1,A wonderful little production. <br /><br />The...,positive,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,positive,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,negative,basically there family little boy jake think t...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter matteis love time money visually stunni...


In [34]:
df=df.drop('review',axis=1)

In [35]:
df.head()

,sentiment,cleaned_review
0,positive,one reviewer mentioned watching 1 oz episode y...
1,positive,wonderful little production filming technique ...
2,positive,thought wonderful way spend time hot summer we...
3,negative,basically there family little boy jake think t...
4,positive,petter matteis love time money visually stunni...


In [36]:
x=df.drop('sentiment',axis=1)

In [37]:
y=df['sentiment']

In [38]:
x_train,x_test,y_train,y_test=train_test_split(x,y,random_state=42,test_size=0.2)

In [39]:
max_features=10000

In [40]:
tokenizer=Tokenizer(num_words=max_features)
tokenizer.fit_on_texts(x_train['cleaned_review'])

x_train_seq=tokenizer.texts_to_sequences(x_train['cleaned_review'])
x_test_seq=tokenizer.texts_to_sequences(x_test['cleaned_review'])

In [41]:
avg_len=sum(len(seq) for seq in x_train_seq)/len(x_train_seq)


In [42]:
avg_len

106.47487709567629

In [43]:
max_len=300

In [45]:
x_train_pad=pad_sequences(x_train_seq,maxlen=max_len,padding='post')
x_test_pad=pad_sequences(x_test_seq,maxlen=max_len,padding='post')

In [46]:
x_train_pad

array([[  14,  317,    1, ...,    0,    0,    0],
       [ 710,   33,    1, ...,    0,    0,    0],
       [ 176,   41,   78, ...,    0,    0,    0],
       ...,
       [ 691,  115,    1, ...,    0,    0,    0],
       [ 237,   92,  556, ...,    0,    0,    0],
       [  65,  248, 1928, ...,    0,    0,    0]], dtype=int32)

In [47]:
y_train

,sentiment
7837,negative
4814,negative
35458,positive
3446,negative
24478,negative
...,...
11304,negative
45059,negative
38405,negative
860,positive


In [48]:
encoder=LabelEncoder()
y_train_encoded=encoder.fit_transform(y_train)
y_test_encoded=encoder.transform(y_test)

In [50]:
y_train=y_train_encoded

In [51]:
y_train

array([0, 0, 1, ..., 0, 1, 1])

In [52]:
y_test=y_test_encoded

In [53]:
rnn=Sequential()

In [54]:
embedding_len=128
max_features=10000
max_length=300

In [55]:
rnn.add(Embedding(input_dim=max_features,output_dim=embedding_len,mask_zero=True))
rnn.add(SimpleRNN(64,return_sequences=False))
rnn.add(Dropout(0.2))
rnn.add(Dense(1,activation='sigmoid'))

In [56]:
rnn.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

In [58]:
rnn.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [61]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history_rnn = rnn.fit(
    x_train_pad, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/5
496/496 ━━━━━━━━━━━━━━━━━━━━ 108s 210ms/step - accuracy: 0.7925 - loss: 0.4408 - val_accuracy: 0.8477 - val_loss: 0.3946
Epoch 2/5
496/496 ━━━━━━━━━━━━━━━━━━━━ 101s 203ms/step - accuracy: 0.9026 - loss: 0.2526 - val_accuracy: 0.8375 - val_loss: 0.3689
Epoch 3/5
496/496 ━━━━━━━━━━━━━━━━━━━━ 105s 212ms/step - accuracy: 0.9567 - loss: 0.1226 - val_accuracy: 0.8341 - val_loss: 0.4443
Epoch 4/5
496/496 ━━━━━━━━━━━━━━━━━━━━ 101s 203ms/step - accuracy: 0.9782 - loss: 0.0623 - val_accuracy: 0.8531 - val_loss: 0.5431
Epoch 5/5
496/496 ━━━━━━━━━━━━━━━━━━━━ 99s 200ms/step - accuracy: 0.9674 - loss: 0.0825 - val_accuracy: 0.8332 - val_loss: 0.5562


In [59]:
lstm_model = Sequential()
lstm_model.add(Embedding(input_dim=max_features, output_dim=embedding_len, mask_zero=True))
lstm_model.add(LSTM(64, return_sequences=False,use_cudnn=False))
lstm_model.add(Dropout(0.2))
lstm_model.add(Dense(1, activation='sigmoid'))

lstm_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [62]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history_lstm = lstm_model.fit(
    x_train_pad, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/5
496/496 ━━━━━━━━━━━━━━━━━━━━ 250s 503ms/step - accuracy: 0.8601 - loss: 0.3379 - val_accuracy: 0.8784 - val_loss: 0.2913
Epoch 2/5
496/496 ━━━━━━━━━━━━━━━━━━━━ 254s 513ms/step - accuracy: 0.9200 - loss: 0.2127 - val_accuracy: 0.8818 - val_loss: 0.3054
Epoch 3/5
496/496 ━━━━━━━━━━━━━━━━━━━━ 250s 505ms/step - accuracy: 0.9448 - loss: 0.1506 - val_accuracy: 0.8751 - val_loss: 0.3200
Epoch 4/5
496/496 ━━━━━━━━━━━━━━━━━━━━ 262s 506ms/step - accuracy: 0.9449 - loss: 0.1471 - val_accuracy: 0.8708 - val_loss: 0.3636
Epoch 5/5
496/496 ━━━━━━━━━━━━━━━━━━━━ 250s 503ms/step - accuracy: 0.9690 - loss: 0.0911 - val_accuracy: 0.8581 - val_loss: 0.4108


In [63]:
gru_model = Sequential()
gru_model.add(Embedding(input_dim=max_features, output_dim=embedding_len, mask_zero=True))
gru_model.add(GRU(64, return_sequences=False,use_cudnn=False))
gru_model.add(Dropout(0.2))
gru_model.add(Dense(1, activation='sigmoid'))

gru_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
gru_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [64]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history_gru = gru_model.fit(
    x_train_pad, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/5
496/496 ━━━━━━━━━━━━━━━━━━━━ 337s 670ms/step - accuracy: 0.8258 - loss: 0.3742 - val_accuracy: 0.8781 - val_loss: 0.3007
Epoch 2/5
496/496 ━━━━━━━━━━━━━━━━━━━━ 305s 515ms/step - accuracy: 0.9151 - loss: 0.2213 - val_accuracy: 0.8883 - val_loss: 0.2849
Epoch 3/5
496/496 ━━━━━━━━━━━━━━━━━━━━ 253s 511ms/step - accuracy: 0.9407 - loss: 0.1616 - val_accuracy: 0.8833 - val_loss: 0.3054
Epoch 4/5
496/496 ━━━━━━━━━━━━━━━━━━━━ 271s 529ms/step - accuracy: 0.9584 - loss: 0.1152 - val_accuracy: 0.8649 - val_loss: 0.3695
Epoch 5/5
496/496 ━━━━━━━━━━━━━━━━━━━━ 259s 521ms/step - accuracy: 0.9763 - loss: 0.0741 - val_accuracy: 0.8762 - val_loss: 0.4205


In [66]:
y_pred_rnn = rnn.predict(x_test_pad)
y_pred_rnn_classes = (y_pred_rnn > 0.5).astype(int)


# Confusion Matrix
print('Confusion Matrix (RNN):')
print(confusion_matrix(y_test, y_pred_rnn_classes))

# Classification Report
print('\nClassification Report (RNN):')
print(classification_report(y_test, y_pred_rnn_classes))


310/310 ━━━━━━━━━━━━━━━━━━━━ 22s 69ms/step
Confusion Matrix (RNN):
[[4123  816]
 [ 746 4232]]

Classification Report (RNN):
              precision    recall  f1-score   support

           0       0.85      0.83      0.84      4939
           1       0.84      0.85      0.84      4978

    accuracy                           0.84      9917
   macro avg       0.84      0.84      0.84      9917
weighted avg       0.84      0.84      0.84      9917



In [67]:
y_pred_lstm = lstm_model.predict(x_test_pad)
y_pred_lstm_classes = (y_pred_rnn > 0.5).astype(int)


# Confusion Matrix
print('Confusion Matrix (LSTM):')
print(confusion_matrix(y_test, y_pred_lstm_classes))

# Classification Report
print('\nClassification Report (LSTM):')
print(classification_report(y_test, y_pred_lstm_classes))

310/310 ━━━━━━━━━━━━━━━━━━━━ 18s 58ms/step
Confusion Matrix (LSTM):
[[4123  816]
 [ 746 4232]]

Classification Report (LSTM):
              precision    recall  f1-score   support

           0       0.85      0.83      0.84      4939
           1       0.84      0.85      0.84      4978

    accuracy                           0.84      9917
   macro avg       0.84      0.84      0.84      9917
weighted avg       0.84      0.84      0.84      9917



In [68]:
y_pred_gru = gru_model.predict(x_test_pad)
y_pred_gru_classes = (y_pred_gru > 0.5).astype(int)


# Confusion Matrix
print('Confusion Matrix (GRU):')
print(confusion_matrix(y_test, y_pred_gru_classes))

# Classification Report
print('\nClassification Report (GRU):')
print(classification_report(y_test, y_pred_gru_classes))

310/310 ━━━━━━━━━━━━━━━━━━━━ 22s 71ms/step
Confusion Matrix (GRU):
[[4439  500]
 [ 648 4330]]

Classification Report (GRU):
              precision    recall  f1-score   support

           0       0.87      0.90      0.89      4939
           1       0.90      0.87      0.88      4978

    accuracy                           0.88      9917
   macro avg       0.88      0.88      0.88      9917
weighted avg       0.88      0.88      0.88      9917



In [69]:
config={
    'max_features':max_features,
    'max_length':max_length,
    'embedding_length':embedding_len
}

### Saving the Best Model (GRU)
Since the GRU model performed the best, we will save the model architecture and weights, along with the tokenizer to process new inputs in the same way.

In [70]:
import pickle

# Save the GRU model
gru_model.save('best_gru_model.keras')

# Save the tokenizer
with open('tokenizer.pkl', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

print("Model and tokenizer saved successfully!")

Model and tokenizer saved successfully!


In [71]:
with open('model_config.pkl', 'wb') as handle:
    pickle.dump(config, handle, protocol=pickle.HIGHEST_PROTOCOL)

print("Configuration saved successfully!")

Configuration saved successfully!
